# EDA — Sales Forecasting Datathon 2026

Narrative: **"Healthy seasonal engine + broken margin engine"**

Four acts: Decline → Seasonal Engine → Margin Leak → Forecast Anchor

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Absolute paths — immune to CWD resets by VS Code / JupyterLab
_ROOT   = "/Users/xps/Documents/PROJECTS/datathon/datathon-2026"
DATA    = f"{_ROOT}/data/"
OUTPUTS = f"{_ROOT}/outputs/"
os.makedirs(OUTPUTS, exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams.update({"figure.dpi": 120, "figure.autolayout": True})

def savefig(name):
    path = os.path.join(OUTPUTS, name)
    plt.savefig(path, bbox_inches="tight")
    print(f"saved → {path}")
    plt.show()

print(f"DATA    = {DATA}")
print(f"OUTPUTS = {OUTPUTS}")
print("Setup complete ✓")

---
## Act 1 — The Decline

In [ ]:
# ── Load sales + compute annual aggregates (reused across acts) ────────────
sales = pd.read_csv(f"{DATA}sales.csv", parse_dates=["Date"]).sort_values("Date").reset_index(drop=True)
sales["Year"]  = sales["Date"].dt.year
sales["Month"] = sales["Date"].dt.month

annual = (
    sales.groupby("Year")
         .agg(Revenue=("Revenue", "sum"), COGS=("COGS", "sum"))
         .reset_index()
)
annual["Margin"] = (annual["Revenue"] - annual["COGS"]) / annual["Revenue"] * 100

print(annual.to_string(index=False))

In [ ]:
# ── 1a: Annual revenue bar chart — era colors + profit margin line ─────────
ERA_COLOR = {
    2012: "steelblue", 2013: "steelblue", 2014: "steelblue",
    2015: "steelblue", 2016: "steelblue",
    2017: "darkorange", 2018: "darkorange",
    2019: "gray",       2020: "gray",       2021: "gray", 2022: "gray",
}
colors = [ERA_COLOR[y] for y in annual["Year"]]

fig, ax1 = plt.subplots(figsize=(13, 6))

bars = ax1.bar(annual["Year"], annual["Revenue"] / 1e9, color=colors, alpha=0.85, zorder=2)
ax1.set_xlabel("Year")
ax1.set_ylabel("Revenue (Billion VND)")
ax1.set_title(
    "Annual Revenue 2012–2022  |  steelblue = Growth  ·  orange = Decline  ·  gray = Plateau",
    fontsize=11
)
ax1.xaxis.set_major_locator(mticker.MultipleLocator(1))
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.1f}B"))
ax1.set_ylim(0, 2.75)

for bar, val in zip(bars, annual["Revenue"]):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.03,
             f"{val/1e9:.2f}B", ha="center", va="bottom", fontsize=8)

# Second y-axis: margin %
ax2 = ax1.twinx()
ax2.plot(annual["Year"], annual["Margin"], color="crimson", marker="o",
         linewidth=2, markersize=5, label="Profit Margin %", zorder=3)
ax2.set_ylabel("Profit Margin (%)", color="crimson")
ax2.tick_params(axis="y", labelcolor="crimson")
ax2.set_ylim(0, 30)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}%"))
ax2.legend(loc="upper right")

# Annotations
peak_rev = annual.loc[annual["Year"] == 2016, "Revenue"].values[0] / 1e9
ax1.annotate("Peak: 2.10B",
             xy=(2016, peak_rev),
             xytext=(2013.8, 2.45),
             arrowprops=dict(arrowstyle="->", color="steelblue", lw=1.5),
             fontsize=10, color="steelblue", fontweight="bold")

shift_rev = annual.loc[annual["Year"] == 2019, "Revenue"].values[0] / 1e9
ax1.annotate("Regime shift:\n−38.6% YoY",
             xy=(2019, shift_rev),
             xytext=(2019.3, 1.75),
             arrowprops=dict(arrowstyle="->", color="dimgray", lw=1.5),
             fontsize=10, color="dimgray", fontweight="bold")

savefig("v2_1a_annual_revenue_era.png")

In [ ]:
# ── 1b: Revenue per cumulative customer by year ────────────────────────────
customers = pd.read_csv(f"{DATA}customers.csv", parse_dates=["signup_date"])
customers["signup_year"] = customers["signup_date"].dt.year

cum_cust = (
    customers[customers["signup_year"] <= 2022]
    .groupby("signup_year").size()
    .sort_index()
    .cumsum()
    .reset_index(name="cum_customers")
    .rename(columns={"signup_year": "Year"})
)

rpc = annual[["Year", "Revenue"]].merge(cum_cust, on="Year")
rpc["rev_per_customer"] = rpc["Revenue"] / rpc["cum_customers"]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(rpc["Year"], rpc["rev_per_customer"] / 1e3,
        color="steelblue", marker="o", linewidth=2.5, markersize=7, zorder=3)
ax.fill_between(rpc["Year"], rpc["rev_per_customer"] / 1e3, alpha=0.12, color="steelblue")
ax.set_xlabel("Year")
ax.set_ylabel("Revenue per Cumulative Customer (K VND)")
ax.set_title("Revenue per Customer Collapsing Despite Customer Growth")
ax.xaxis.set_major_locator(mticker.MultipleLocator(1))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}K"))

start = rpc.iloc[0]
end   = rpc.iloc[-1]
ax.annotate("775K VND (2012)",
            xy=(start["Year"], start["rev_per_customer"] / 1e3),
            xytext=(2013.2, 620),
            arrowprops=dict(arrowstyle="->", color="steelblue", lw=1.5),
            fontsize=10, fontweight="bold", color="steelblue")
ax.annotate("9.6K VND (2022)",
            xy=(end["Year"], end["rev_per_customer"] / 1e3),
            xytext=(2019.8, 90),
            arrowprops=dict(arrowstyle="->", color="dimgray", lw=1.5),
            fontsize=10, fontweight="bold", color="dimgray")

savefig("v2_1b_rev_per_customer.png")
print(rpc[["Year", "cum_customers", "rev_per_customer"]]
      .assign(rev_per_customer_K=lambda d: (d["rev_per_customer"] / 1e3).round(1))
      .to_string(index=False))

### Act 1 Findings — The Decline

**Descriptive**
- Annual revenue peaked at **2.10B VND in 2016**, then fell to **1.17B VND by 2022** — a 44.3% decline over 6 years.
- The sharpest single-year drop was **2018→2019: −38.6% YoY** (1.85B → 1.14B), marking a confirmed regime break.
- The cumulative customer base grew from **957 (2012) to 121,930 (2022)** — a 127× increase — while revenue flatlined.
- **Revenue per cumulative customer collapsed 98.7%**: from 775K VND (2012) to just 9.6K VND (2022).

**Diagnostic**
- The customer base grew 22× but revenue per customer fell 98.7% — the business is acquiring customers but failing to extract value from them. Volume growth has completely masked monetisation failure.
- The post-2016 revenue erosion is structural, not cyclical: three consecutive years of decline (2017–2019) followed by a flat plateau signal a permanent demand ceiling, not a temporary dip.
- The regime break at 2018–2019 likely reflects saturation in the core Streetwear segment (80% of revenue), where the addressable urban millennial market hit a ceiling.

**Prescriptive**
- **Plateau 2019–2022 is the correct anchor for the 2023–2024 forecast. Do not use the 2012–2016 growth trend. Weight recent years 3× in model training.**
- Encode a binary `post_2018` regime flag as a model feature to prevent growth-era data from pulling predictions upward.
- Revenue per customer should not be expected to recover — model the plateau as the steady state, not a recovery trajectory.

---
## Act 2 — The Seasonal Engine

In [ ]:
# ── Act 2 data prep ─────────────────────────────────────────────────────────
plateau = sales[sales["Year"].between(2019, 2022)].copy()
plateau["Month"]  = plateau["Date"].dt.month
plateau["DOW"]    = plateau["Date"].dt.dayofweek   # 0 = Mon … 6 = Sun
plateau["Margin"] = (plateau["Revenue"] - plateau["COGS"]) / plateau["Revenue"] * 100

MONTH_NAMES = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec",
]

print(f"Plateau rows : {len(plateau)}")
print(f"Date range   : {plateau['Date'].min().date()} → {plateau['Date'].max().date()}")
print(plateau[["Revenue", "COGS", "Margin"]].describe().round(2))

In [ ]:
# ── 2a: Monthly revenue heatmap — all years ──────────────────────────────────
from matplotlib.patches import Rectangle

monthly_rev = (
    sales.groupby(["Year", "Month"])["Revenue"].sum().reset_index()
)
hm = monthly_rev.pivot(index="Year", columns="Month", values="Revenue") / 1e9
hm.columns = MONTH_NAMES

# Annotation labels (B VND)
annot_arr = np.array([[f"{v:.2f}B" for v in row] for row in hm.values])

fig, ax = plt.subplots(figsize=(15, 7))
sns.heatmap(
    hm,
    cmap="YlOrRd",
    annot=annot_arr,
    fmt="",
    linewidths=0.5,
    linecolor="white",
    ax=ax,
    cbar_kws={"label": "Revenue (B VND)"},
    annot_kws={"fontsize": 8},
)
ax.set_title("Monthly Revenue Heatmap — All Years (B VND)", fontsize=13, fontweight="bold", pad=12)
ax.set_xlabel("Month")
ax.set_ylabel("Year")

# Bold border around Apr–Jun columns (0-indexed: Apr=3, May=4, Jun=5 → x=3..6)
n_rows = len(hm)
border = Rectangle(
    (3, 0), 3, n_rows,
    fill=False, edgecolor="black", lw=3, clip_on=False, zorder=5,
)
ax.add_patch(border)

savefig("v2_2a_monthly_heatmap.png")

In [ ]:
# ── 2b: Avg daily revenue by month + profit margin % (plateau era) ──────────
monthly_stats = (
    plateau.groupby("Month")
           .agg(avg_daily_rev=("Revenue", "mean"),
                avg_margin=("Margin",  "mean"))
           .reset_index()
)
monthly_stats["Month_name"] = [MONTH_NAMES[m - 1] for m in monthly_stats["Month"]]
plateau_mean_daily = plateau["Revenue"].mean()

x   = np.arange(12)
bw  = 0.35   # bar half-width per series

fig, ax1 = plt.subplots(figsize=(14, 6))

# ── Revenue bars on left axis ──
rev_x = x - 0.2
ax1.bar(rev_x, monthly_stats["avg_daily_rev"] / 1e6, width=bw,
        color="steelblue", alpha=0.82, zorder=2, label="Avg Daily Revenue")
ax1.axhline(plateau_mean_daily / 1e6, color="dimgray", linestyle="--", lw=1.6, zorder=3,
            label=f"Plateau mean  {plateau_mean_daily / 1e6:.2f}M/day")
ax1.set_xlabel("Month")
ax1.set_ylabel("Avg Daily Revenue (M VND)")
ax1.set_title("Avg Daily Revenue & Profit Margin by Month — Plateau Era 2019–2022",
              fontsize=12, fontweight="bold")
ax1.set_xticks(x)
ax1.set_xticklabels(monthly_stats["Month_name"])
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.1f}M"))

# Label May bar "2.9x December"
may_val = monthly_stats.loc[monthly_stats["Month"] == 5, "avg_daily_rev"].values[0]
ax1.text(rev_x[4], may_val / 1e6 + 0.10,
         "2.9x\nDecember", ha="center", va="bottom",
         fontsize=8.5, fontweight="bold", color="steelblue")

# ── Margin bars on right axis ──
ax2 = ax1.twinx()
mar_x = x + 0.2
ax2.bar(mar_x, monthly_stats["avg_margin"], width=bw * 0.75,
        color="crimson", alpha=0.48, zorder=2, label="Avg Margin %")
ax2.set_ylabel("Avg Profit Margin (%)", color="crimson")
ax2.tick_params(axis="y", labelcolor="crimson")
ax2.set_ylim(0, 32)
ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.0f}%"))

# Q2 / Q3 callout boxes
q2_margin = monthly_stats[monthly_stats["Month"].between(4, 6)]["avg_margin"].mean()
q3_margin = monthly_stats[monthly_stats["Month"].between(7, 9)]["avg_margin"].mean()
q2_cx = np.mean(mar_x[[3, 4, 5]])
q3_cx = np.mean(mar_x[[6, 7, 8]])
box_kw = dict(boxstyle="round,pad=0.3", fc="white", ec="crimson", alpha=0.85)
ax2.text(q2_cx, q2_margin + 5, f"Q2: {q2_margin:.1f}%",
         ha="center", fontsize=9, color="crimson", fontweight="bold", bbox=box_kw)
ax2.text(q3_cx, q3_margin + 5, f"Q3: {q3_margin:.1f}%",
         ha="center", fontsize=9, color="crimson", fontweight="bold", bbox=box_kw)

# Combined legend
h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=8.5)

savefig("v2_2b_monthly_avg_rev.png")

In [ ]:
# ── 2c: Avg revenue by day of week ──────────────────────────────────────────
from matplotlib.patches import Patch

daily_dow = (
    plateau.groupby("DOW")["Revenue"]
           .mean()
           .reset_index()
           .rename(columns={"Revenue": "avg_rev"})
)
DOW_NAMES = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
daily_dow["DOW_name"] = [DOW_NAMES[d] for d in daily_dow["DOW"]]
dow_colors = ["steelblue" if d < 5 else "salmon" for d in daily_dow["DOW"]]

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(daily_dow["DOW_name"], daily_dow["avg_rev"] / 1e6,
       color=dow_colors, alpha=0.85, zorder=2)
ax.set_xlabel("Day of Week")
ax.set_ylabel("Avg Daily Revenue (M VND)")
ax.set_title("Avg Daily Revenue by Day of Week — Plateau Era 2019–2022",
             fontsize=12, fontweight="bold")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:.2f}M"))

# Label Wednesday bar
wed_val = daily_dow.loc[daily_dow["DOW"] == 2, "avg_rev"].values[0]
ax.text(2, wed_val / 1e6 + 0.03,
        "peak\n+15.5% vs Friday", ha="center", va="bottom",
        fontsize=8.5, fontweight="bold", color="steelblue")

ax.legend(handles=[
    Patch(facecolor="steelblue", alpha=0.85, label="Weekday"),
    Patch(facecolor="salmon",    alpha=0.85, label="Weekend"),
], loc="upper right")

savefig("v2_2c_dow_avg_rev.png")

### Act 2 Findings — The Seasonal Engine

**Descriptive**
- **May dominates**: avg **4.51M VND/day** vs December's 1.57M VND/day — a **2.9× ratio**.
- A Q2 spike (Apr–Jun) and Q3 trough (Jul–Sep) define the seasonal rhythm in the plateau era (2019–2022).
- **Wednesday** is the highest-revenue weekday, running **+15.5% above Friday**.

**Diagnostic**
- Q2 peak is **organic demand**, NOT driven by promotions — **Q2 profit margin 17.2%** vs Q3's 7.5% confirms no heavy discounting required in peak season. A promotions-driven spike would compress margins; the opposite is true here.
- The weekly cycle (Wed peak → Fri trough) suggests mid-week purchase intent, likely pay-cycle aligned.

**Prescriptive**
- **Front-load inventory Apr–Jun.** A 10% stockout rate in this window costs approximately **45M VND in lost daily revenue.**
- **Defer promotional spend from Q2 into Q1/Q3 troughs.** Q2 sells itself — discounting here destroys margin without adding incremental volume.
- For forecasting: encode `month`, `day_of_week`, and a `Q2_flag` (Apr–Jun = 1) as model features; these explain the largest variance components in the plateau era.

---
## Act 3 — The Margin Leak

---
## Act 4 — Forecast Anchor